# 10 • Tool Calling: ตรวจสถานการณ์คลังยาง

เป้าหมาย: ดูชื่อ tool และ arguments ที่ LLM เลือก รัน Python tool แล้วส่งผลกลับไปให้ LLM ตอบภาษาไทย

**ข้อมูลสมมติสำหรับการอบรม:** ชื่อหน่วยงาน Lot ผู้ขาย ค่าที่วัด และ กฏ ในตัวอย่างถูกสร้างขึ้นเพื่อเรียนรู้ AI ไม่ใช่ข้อมูลการผลิตหรือระเบียบจริงของกลุ่มศรีตรัง

[เปิด notebook นี้ใน Colab](https://colab.research.google.com/drive/1yJodowG5kF_ay4JmDTSv0SsQAKYmlJwn?usp=drive_link)


## เตรียมพร้อมก่อนเริ่ม
แทนที่ `your_api_key` ในตัวแปร `OPENAI_API_KEY` ด้วย API key จริงของคุณ แล้วรัน cells จากบนลงล่าง ใช้รูปแบบเดียวกันทั้ง local และ Colab โดย cell ติดตั้ง packages อยู่ใน notebook แล้ว

ก่อนแชร์หรือ commit notebook ให้เปลี่ยน API key กลับเป็น `your_api_key`


In [ ]:
%pip install -q openai

In [ ]:
from openai import OpenAI

OPENAI_API_KEY = "your_api_key"  # แทนที่ด้วย API key จริงของคุณ
client = OpenAI(api_key=OPENAI_API_KEY)
MODEL = "gpt-4.1-mini"


In [ ]:
def get_flood_level(factory):
    readings = {"คลังยางแท่ง A": 55, "คลังน้ำยาง B": 10}
    if factory not in readings:
        return {"error": "ไม่พบหน่วยงานนี้", "source": "ข้อมูลสมมติสำหรับการอบรม"}
    return {"factory": factory, "water_cm": readings[factory], "source": "ข้อมูลสมมติสำหรับการอบรม", "live": False}

In [ ]:
tools = [{
    "type": "function", "name": "get_flood_level",
    "description": "อ่านระดับน้ำสมมติของคลังยางแท่ง A หรือคลังน้ำยาง B",
    "parameters": {"type": "object", "properties": {
        "factory": {"type": "string", "enum": ["คลังยางแท่ง A", "คลังน้ำยาง B"]}
    }, "required": ["factory"], "additionalProperties": False},
    "strict": True,
}]

In [ ]:
import json

messages = [{"role": "user", "content": "ระดับน้ำสมมติที่คลังยางแท่ง A เป็นเท่าไร?"}]
response = client.responses.create(
    model=MODEL, input=messages, tools=tools,
    tool_choice={"type": "function", "name": "get_flood_level"},
    max_output_tokens=500,
)
messages += response.output
for call in response.output:
    if call.type == "function_call":
        print("Tool ที่เลือก:", call.name, "Arguments:", call.arguments)
        result = get_flood_level(**json.loads(call.arguments))
        print("ผลลัพธ์:", result)
        messages.append({"type": "function_call_output", "call_id": call.call_id,
                         "output": json.dumps(result, ensure_ascii=False)})

In [ ]:
final = client.responses.create(
    model=MODEL, input=messages,
    instructions="ตอบเป็นภาษาไทยและระบุชัดเจนว่าค่าที่อ่านเป็นข้อมูลสมมติสำหรับการอบรม",
    max_output_tokens=500,
)
print(final.output_text)

## ลองทำด้วยตนเอง
1. ถามเกี่ยวกับคลังน้ำยาง B และดู arguments
2. เปลี่ยนเป็น tool_choice='auto' แล้วลองทักทายแทนคำถามสถานะ
3. อธิบายว่าส่วนใดเลือก tool และส่วนใดรัน Python

ค่าระดับน้ำเป็น sample data ไม่ใช่สถานการณ์จริง
